# 06 · End-to-end pipeline and reproducibility audit

Notebooks 01–05 start from the archived outputs of the HPC runs. This notebook documents how those outputs were produced and checks that the archived files are the ones the HPC wrote.

| Stage | Script(s) in `src/hpc_pipeline/` | Needs | Output used here |
|---|---|---|---|
| 1. Cohort, split, training | `01_data_and_training/build_final_1d.py`, `build_frozen_split.py`, `train_1d_author_code.py`, `train_2d_author_code.py` | raw ECG CSVs, rendered images, GPU | `data/archived/split/`, `replication/` |
| 2. Grad-CAM, alignment, Module 1 | `02_gradcam_and_module1/run_gradcam_*_author_code.py`, `run_dan_module1_*.py`, `module1_alignment_branch/` | trained models | `module1/` |
| 3. Module 2 stability (run on Kaggle) | `03_module2_stability/run_dan_module2_noise_stability.py` | models, ECGs, renderer | `module2/` |
| 4. Module 3A / 3B | `04_module3_fusion_faithfulness/m3a_prepare.py`, `m3b_generate_trace_support.py`, `m3b_apply.py` | Module 2 output, 2D model | `module3a/`, `module3b/`, `module3b_sensitivity/` |
| 5. Deletion faithfulness | `module3_faithfulness.py` (SLURM array 0–692), `summarize_692_exploratory.py` | models, ECGs, renderer | `faithfulness_cohort/` |

Stages 1–5 need the Chapman–Shaoxing data, the trained `.keras` models and TensorFlow. Module 2 ran on Kaggle (TensorFlow 2.21); the other stages ran on the KFUPM HPC (CPU partition, TensorFlow 2.19 with oneDNN disabled for Module 3).
Set `RUN_ON_HPC = True` and run this notebook on the HPC to execute a short smoke test of stages 4–5 for ECG 209.

> **Code on GitHub.** The re-run of the MM-GradCAM anchor (cohort, frozen split, 1D/2D CNN training and test-split manifests) is on the `main` branch of <https://github.com/dwiharyani/dan-mmgradcam-anchor-replication>. The Grad-CAM temporal alignment and the Module 1 alignment code are on the `module1-nahida` branch of the same repository. The same scripts are also included in `src/hpc_pipeline/` of the submitted package.

> **Google Colab:** upload `DAN-MMGradCAM_COE691.zip` (Files panel), then run `!unzip -q DAN-MMGradCAM_COE691.zip -d /content` once in a code cell. The setup cell below finds the folder automatically. **Jupyter:** open this notebook from the `notebooks/` folder of the extracted package.

In [1]:
import sys, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

def find_root():
    """Repository root: works from notebooks/, from the root itself, and in Google Colab (/content)."""
    here = Path.cwd().resolve()
    for p in [here, *here.parents, Path('/content/DAN-MMGradCAM'), Path('/content/drive/MyDrive/DAN-MMGradCAM')]:
        if (p / 'data' / 'archived').is_dir() and (p / 'src').is_dir():
            return p
    raise FileNotFoundError('DAN-MMGradCAM folder not found. In Colab, upload DAN-MMGradCAM_COE691.zip and run: '
                            '!unzip -q DAN-MMGradCAM_COE691.zip -d /content')
ROOT = find_root()                          # repository root
sys.path.insert(0, str(ROOT / 'src'))
ARCH = ROOT / 'data' / 'archived'           # archived HPC outputs (inputs of this notebook)
TAB = ROOT / 'results' / 'tables'           # tables written by the notebooks
FIG = ROOT / 'results' / 'figures'          # figures written by the notebooks (used in the report)
TAB.mkdir(parents=True, exist_ok=True); FIG.mkdir(parents=True, exist_ok=True)
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 40); pd.set_option('display.precision', 4)
warnings.filterwarnings('ignore', category=RuntimeWarning)
CLASSES = ['AFib', 'GSVT', 'SB', 'SR']
def show(name, width=900):
    """Display a figure that a plotting function saved to results/figures."""
    display(Image(filename=str(FIG / f'{name}.png'), width=width))
print('repository root:', ROOT.name)

repository root: DAN-MMGradCAM


In [2]:
RUN_ON_HPC = False                     # set True on the HPC, inside ~/mm_gradcam with the project .venv active
HPC_BASE = Path.home() / 'mm_gradcam' if RUN_ON_HPC else Path('~/mm_gradcam')   # project folder on the HPC
import platform, sklearn, scipy, matplotlib
print('python', platform.python_version(), '| numpy', np.__version__, '| pandas', pd.__version__, '| scipy', scipy.__version__,
      '| scikit-learn', sklearn.__version__, '| matplotlib', matplotlib.__version__)

python 3.13.18 | numpy 2.5.3 | pandas 3.0.5 | scipy 1.18.1 | scikit-learn 1.9.1 | matplotlib 3.11.2


## 1. Integrity of the archived files against the HPC SHA-256 manifest

In [3]:
import hashlib
manifest = json.loads((ARCH / 'provenance/hpc_manifest_sha256.json').read_text())
by_name = {}
for k, v in manifest.items():
    by_name.setdefault(Path(k).name, []).append(v['sha256'])
rows = []
for p in sorted(ARCH.rglob('*')):
    if p.is_file() and p.name in by_name and 'provenance' not in p.parts:
        h = hashlib.sha256(p.read_bytes()).hexdigest()
        rows.append({'file': str(p.relative_to(ARCH)), 'match': h in by_name[p.name]})
chk = pd.DataFrame(rows)
print(f'archived files listed in the HPC manifest: {len(chk)} | SHA-256 match: {int(chk.match.sum())} | mismatch: {int((~chk.match).sum())}')
chk.groupby(chk.file.str.split('/').str[0]).match.agg(['count', 'sum']).rename(columns={'count': 'files', 'sum': 'matching'})

archived files listed in the HPC manifest: 1399 | SHA-256 match: 1399 | mismatch: 0


,files,matching
file,,
faithfulness_cohort,1384,1384
module2,3,3
module3a,6,6
module3b,3,3
module3b_sensitivity,2,2
split,1,1


In [4]:
mb = json.loads((ARCH / 'provenance/module1_branch_files.json').read_text())
ok = {k: hashlib.sha256((ARCH / 'module1' / 'branch' / k).read_bytes()).hexdigest() == v['sha256'] for k, v in mb['files'].items()}
print(f"Module 1 branch files (branch {mb['source_branch']}, commit {mb['source_commit']}): {sum(ok.values())} of {len(ok)} match their recorded SHA-256")

Module 1 branch files (branch module1-nahida, commit f5504c1): 5 of 5 match their recorded SHA-256


## 2. Run configuration recorded by the HPC

In [5]:
m2 = json.loads((ARCH / 'module2/run_metadata.json').read_text())
cfg = m2['config']
print('Module 2: records', m2['n_records'], '| TensorFlow', m2['tf'], '| scaler fitted on', m2['scaler_n_fit'], 'recordings')
print('  scales', cfg['SCALES'], '| drift', cfg['DRIFT_HZ'], 'Hz', cfg['DRIFTS'], 'x', len(cfg['DRIFT_PHASES']), 'phases',
      '| noise', cfg['NOISES'], 'x', cfg['NOISE_SEEDS'], 'seeds')
print('  waveform columns', cfg['X0'], '-', cfg['X1'], '| grid', cfg['W'], '| corridor ±', cfg['MASK_DILATE_Y'], 'px | min accepted', cfg['MIN_ACCEPTED'])
a0 = json.loads((ARCH / 'faithfulness_cohort/raw/faith_0000_audit.json').read_text())
print('Faithfulness: block', a0['block_samples'], 'samples | fractions', a0['fractions'], '| random repeats', a0['random_repeats'],
      '| seed', a0['seed'], '| TensorFlow', a0['tensorflow_version'], '| oneDNN disabled', a0['oneDNN_disabled'])

Module 2: records 1054 | TensorFlow 2.21.0 | scaler fitted on 9534 recordings
  scales [0.8, 0.9, 0.95, 1.05, 1.1, 1.2] | drift 0.3 Hz [0.05, 0.1] x 6 phases | noise [0.05, 0.1] x 6 seeds
  waveform columns 58 - 1021 | grid 600 | corridor ± 6 px | min accepted 2
Faithfulness: block 50 samples | fractions [0.05, 0.1, 0.15, 0.2, 0.25, 0.3] | random repeats 5 | seed 20261115 | TensorFlow 2.19.0 | oneDNN disabled True


## 3. Documented reproduction failures

In [6]:
for f in sorted((ARCH / 'provenance/diagnostics_ecg406').glob('m3faith_*602.log')):
    hits = [l.strip() for l in f.read_text(errors='ignore').splitlines() if 'QC mismatch' in l or 'baseline' in l.lower()]
    print(f.name, '->', hits[-1] if hits else 'no QC line')
import re
for f in sorted((ARCH / 'provenance/diagnostics_ecg406').glob('m3b_debug406_*.log')):
    txt = f.read_text(errors='ignore')
    mode = re.search(r'MODE=(\w+)', txt).group(1); err = float(re.search(r'DBG406 ERROR: ([0-9.e-]+)', txt).group(1))
    print(f'renderer {mode:5s}: 2D baseline probability error {err:.2e} -> {"PASS" if "PASS" in txt else "FAIL"} (tolerance 1e-5)')
prov = json.loads((ARCH / 'provenance/hpc_provenance.json').read_text())
print({k: prov[k] for k in ['faithfulness_success', 'faithfulness_failed_ecg', 'module3a_eligible', 'module3b_strict', 'module3b_sensitivity',
                             'model_checkpoints_included', 'raw_ecg_included']})

m3faith_2409801_602.log -> RuntimeError: 2D baseline QC mismatch ECG 406: 5.6266785e-05
m3faith_2410627_602.log -> RuntimeError: 2D baseline QC mismatch ECG 406: 5.6266785e-05
renderer fresh: 2D baseline probability error 5.63e-05 -> FAIL (tolerance 1e-5)
renderer warm : 2D baseline probability error 2.38e-07 -> PASS (tolerance 1e-5)
{'faithfulness_success': 692, 'faithfulness_failed_ecg': 406, 'module3a_eligible': 694, 'module3b_strict': 693, 'module3b_sensitivity': 694, 'model_checkpoints_included': False, 'raw_ecg_included': False}


ECG 406 failed the strict 2D baseline check because the matplotlib renderer keeps state between calls: a fresh renderer and a renderer that had drawn other recordings
produced different image bytes. The 2D deletion results are therefore reported as exploratory, and ECG 406 is excluded rather than imputed.

## 4. Smoke test on the HPC (stages 4–5 for one recording)

In [7]:
import subprocess, shlex
M3 = HPC_BASE / 'module3_aligned'; SRC = HPC_BASE / 'module3_sources'
M3A_DIR = M3 / 'outputs' / 'module3a_v1'          # Module 3A output folder as named on the HPC
steps = [
    f"python src/hpc_pipeline/04_module3_fusion_faithfulness/m3b_generate_trace_support.py --m2-dir {M3}/inputs/module2_s9534 "
    f"--m3a-dir {M3A_DIR} --runtime-root {SRC}/module2_runtime --module2-common src/hpc_pipeline/03_module2_stability/module2_common.py "
    f"--only-ids 209 --out results/smoke/trace_209.npz",
    "python src/hpc_pipeline/04_module3_fusion_faithfulness/module3_faithfulness.py --index 0 --out-dir results/smoke/faith",
]
if RUN_ON_HPC:
    for cmd in steps:
        print('$', cmd); out = subprocess.run(shlex.split(cmd), cwd=ROOT, capture_output=True, text=True)
        print(out.stdout[-2000:], out.stderr[-2000:])
else:
    print('RUN_ON_HPC is False: the commands below are printed, not executed (they need the models, ECGs and TensorFlow).')
    for cmd in steps:
        print('$', cmd)

RUN_ON_HPC is False: the commands below are printed, not executed (they need the models, ECGs and TensorFlow).
$ python src/hpc_pipeline/04_module3_fusion_faithfulness/m3b_generate_trace_support.py --m2-dir ~/mm_gradcam/module3_aligned/inputs/module2_s9534 --m3a-dir ~/mm_gradcam/module3_aligned/outputs/module3a_v1 --runtime-root ~/mm_gradcam/module3_sources/module2_runtime --module2-common src/hpc_pipeline/03_module2_stability/module2_common.py --only-ids 209 --out results/smoke/trace_209.npz
$ python src/hpc_pipeline/04_module3_fusion_faithfulness/module3_faithfulness.py --index 0 --out-dir results/smoke/faith
